# Task 1.3 — What the Paper Claims to Improve
**Paper:** Poisoning Attacks against Support Vector Machines  
**Authors:** Battista Biggio, Blaine Nelson, Pavel Laskov (ICML 2012)  
**Roll Number:** 230110  
**Random Seed:** N/A (no code in this notebook)

---
## Main Baseline / Prior Method

The paper primarily compares against **feature-space adversarial attacks** on learning algorithms, specifically the methods of **Brückner & Scheffer (2009)** and **Kloft & Laskov (2010)**.

Brückner & Scheffer (2009) formulate adversarial interactions as a Nash equilibrium game between learner and adversary but construct attack points *in the kernel feature space* $\mathcal{H}$, not the original input space $\mathbb{R}^d$. Kloft & Laskov (2010) study online anomaly detection under adversarial impact, again requiring the adversary to operate in feature space.

For random label-flip baselines, the paper also implicitly compares against **random label flipping** — injecting a randomly chosen point with a flipped label — showing that their gradient-based attack achieves significantly higher error rates (Figure 2, rightmost plots: gradient attack produces 15–20% error vs. 2–5% from initial random flip).

---
## Limitation of the Baseline That the Paper Identifies

The key limitation the paper identifies in the prior work (Brückner & Scheffer, 2009; Kloft & Laskov, 2010) is that these methods **require the attacker to construct adversarial points in the kernel feature space** $\phi(x)$, rather than in the original input space $\mathbb{R}^d$.

For non-linear kernels (e.g., RBF, polynomial), the feature space $\mathcal{H}$ is typically infinite-dimensional and the attacker has no practical way to access it. The attacker lives in the real world and must craft data in the input space — for example, an actual image, email, or network packet. If the attack is only expressible as a direction in $\mathcal{H}$, there is no inverse mapping $\phi^{-1}$ to convert it back to a concrete real-world data point. This makes prior attacks *theoretically valid but practically infeasible* for non-linear kernels.

As the paper states (Section 1): *"such influence [on functions in the RKHS] may facilitate the practical realization of various evasion strategies"* — contrasting their kernelised gradient approach against the prior feature-space limitation.

---
## How the Proposed Method Overcomes This Limitation

Biggio et al. derive the gradient of the attacker's objective function (Eq. 10) such that it depends on the attack point $x_c$ **only through the kernel function gradients** $\partial K(x_i, x_c) / \partial u$. Because standard kernels (linear, polynomial, RBF) all have closed-form derivatives with respect to input-space coordinates (Section 2.2), the entire attack can be computed and executed **entirely in the original input space** $\mathbb{R}^d$ — even when using non-linear kernels.

This is achieved by exploiting the implicit function theorem on the SVM's KKT conditions: the attacker differentiates through the SVM's optimal solution using the adiabatic update (Eqs. 4–9), which yields a closed-form expression for how the SVM's dual variables and bias respond to a change in $x_c$. The resulting gradient is fully kernelised, meaning no explicit computation in $\mathcal{H}$ is required.

---
## Scenario Where the Paper's Method Would Not Outperform the Baseline

The paper's method would **not** outperform the feature-space baseline when a **linear kernel** is used.

With a linear kernel, the feature space $\mathcal{H} = \mathbb{R}^d$ is identical to the input space. The kernel function is simply $K(x_i, x_j) = x_i^\top x_j$, and the feature mapping is the identity: $\phi(x) = x$. In this scenario, the limitation that the paper identifies — the infeasibility of constructing attacks in feature space — **disappears entirely**. Both the prior methods and the proposed method operate in the same space, and the baseline methods of Brückner & Scheffer or Kloft & Laskov can construct their attack points directly in $\mathbb{R}^d$ without any inverse mapping problem.

Furthermore, for linear kernels, the SVM's decision surface is a single hyperplane, making the optimisation landscape relatively simple. In this regime, even a random label-flip attack may cause meaningful damage (as shown in the paper's Figure 1, top row), and the marginal benefit of the more sophisticated KKT-gradient approach — which carries computational overhead from maintaining the incremental SVM solution and matrix inversions at each step — may not justify its complexity.

Additionally, if the attacker has access to a surrogate model rather than the exact training data (violating Assumption 1 from Task 1.2), the gradient computed via KKT conditions would be noisy and may not converge to effective attack points. In contrast, simpler feature-space heuristics might be more robust to model mismatch because they do not rely on exact KKT differentiation.

This scenario is supported by the paper's own experimental design, which primarily showcases the method's advantage in the non-linear (RBF) kernel setting — the linear kernel results (Figure 1, top row) show a more straightforward attack surface that any gradient-based method could exploit.